# 🎓 NLP Mini Project: Sentiment Classification Pipeline
### Complete Pipeline: Preprocessing ➔ TF-IDF ➔ Logistic Regression ➔ Evaluation

This notebook trains a high-accuracy, lightweight text sentiment classification model on the IMDb benchmark dataset. The trained model (`sentiment_model.pkl`) and vectorizer (`tfidf_vectorizer.pkl`) can be downloaded at the end and placed inside the `models/` directory for the interactive Streamlit app.

## 1. Setup & Dependencies

In [ ]:
!pip install -q scikit-learn nltk datasets pandas numpy joblib matplotlib seaborn

import re
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
print('All libraries and NLTK resources loaded successfully!')

## 2. Load Dataset (IMDb Benchmark)

In [ ]:
print('Loading IMDb dataset from HuggingFace...')
dataset = load_dataset('stanfordnlp/imdb')

df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])

# Sample for swift training
SAMPLE_SIZE = 20000
df_train = df_train.groupby('label', group_keys=False).apply(lambda x: x.sample(SAMPLE_SIZE // 2, random_state=42)).reset_index(drop=True)
df_test = df_test.groupby('label', group_keys=False).apply(lambda x: x.sample(5000, random_state=42)).reset_index(drop=True)

print(f'Training samples: {len(df_train)} (Balanced: {df_train.label.value_counts().to_dict()})')
print(f'Testing samples: {len(df_test)}')
df_train.head()

## 3. NLP Preprocessing Pipeline
We clean raw HTML, convert to lowercase, remove punctuation, filter non-sentiment stopwords, and lemmatize tokens.

In [ ]:
stop_words = set(stopwords.words('english'))
negation_words = {'not', 'no', 'nor', 'neither', 'never', 'hardly', 'barely', "isn't", "wasn't", "didn't", "don't", "won't", "can't", "couldn't"}
custom_stopwords = stop_words - negation_words
lemmatizer = WordNetLemmatizer()

def clean_text(raw_text):
    lowered = raw_text.lower()
    no_html = re.sub(r'<.*?>', ' ', lowered)
    no_urls = re.sub(r'http\S+|www\S+', '', no_html)
    cleaned_chars = re.sub(r"[^a-z\s']", ' ', no_urls)
    tokens = cleaned_chars.split()
    processed = [lemmatizer.lemmatize(t.strip("'")) for t in tokens if len(t.strip("'")) > 1 and t.strip("'") not in custom_stopwords]
    return ' '.join(processed)

print('Preprocessing training data...')
t0 = time.time()
df_train['clean_text'] = df_train['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)
print(f'Done in {time.time() - t0:.2f} seconds!')

## 4. Feature Extraction: TF-IDF Vectorizer
We capture both individual words (unigrams) and phrases (bigrams) like `'not good'` or `'highly recommended'`.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3
)

X_train = vectorizer.fit_transform(df_train['clean_text'])
X_test = vectorizer.transform(df_test['clean_text'])
y_train = df_train['label'].values
y_test = df_test['label'].values

print(f'TF-IDF Matrix shape: {X_train.shape}')

## 5. Model Training: Tuned Logistic Regression

In [ ]:
model = LogisticRegression(C=2.5, max_iter=1000, solver='lbfgs', random_state=42)
t0 = time.time()
model.fit(X_train, y_train)
print(f'Training completed in {time.time() - t0:.2f} seconds!')

## 6. Model Evaluation & Confusion Matrix

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc * 100:.2f}%\n')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 7. Model Explainability: Top Sentiment Keywords

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]

top_positive = feature_names[np.argsort(coefficients)[-10:]][::-1]
top_negative = feature_names[np.argsort(coefficients)[:10]]

print('🌟 Top 10 Words Indicating Positive Sentiment:', list(top_positive))
print('⚠️ Top 10 Words Indicating Negative Sentiment:', list(top_negative))

## 8. Export Model Artifacts for Streamlit App

In [ ]:
import os
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/sentiment_model.pkl')
joblib.dump(vectorizer, 'models/tfidf_vectorizer.pkl')
print('Saved models/sentiment_model.pkl and models/tfidf_vectorizer.pkl successfully!')

# If running in Google Colab, trigger direct download:
try:
    from google.colab import files
    files.download('models/sentiment_model.pkl')
    files.download('models/tfidf_vectorizer.pkl')
except ImportError:
    pass